# SICR and staging

Staging combines lifetime-PD deterioration, delinquency, the 30-DPD backstop, modification history and default.

In [1]:
from pathlib import Path
import json
import sqlite3
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import roc_auc_score, brier_score_loss, mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd().resolve()
if not (ROOT / "config").exists():
    ROOT = ROOT.parent
DB = ROOT / "database" / "ifrs9_ecl.sqlite3"
CFG = yaml.safe_load((ROOT / "config" / "project.yaml").read_text())

def query(sql):
    with sqlite3.connect(DB) as connection:
        return pd.read_sql_query(sql, connection)

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
plt.rcParams["figure.figsize"] = (9, 4)

Stage 3 is assigned first. Stage 2 is assigned when any configured SICR trigger applies. Remaining loans are Stage 1. Current and origination lifetime PD are calculated from the monthly hazard over the same remaining horizon, so the comparison measures risk deterioration rather than a horizon difference.

In [2]:
{'relative_lifetime_pd_multiple':CFG['sicr']['relative_pd_multiple'],
 'absolute_lifetime_pd_increase':CFG['sicr']['absolute_pd_increase'],
 'dpd_backstop':CFG['sicr']['dpd_backstop']}

{'relative_lifetime_pd_multiple': 2.0,
 'absolute_lifetime_pd_increase': 0.02,
 'dpd_backstop': 30}

In [3]:
query('select * from stage_summary order by stage')

,stage,loans,gross_exposure,ecl,coverage_ratio,exposure_share,ecl_share
0,1,241,"36,407,043.1800","1,387.6591",0.0000,0.0841,0.0041
1,2,3212,"393,794,048.4600","304,151.4746",0.0008,0.9095,0.8925
2,3,18,"2,762,680.3300","35,228.2766",0.0128,0.0064,0.1034


In [4]:
query('''select stage, primary_stage_reason, count(*) loans,
sum(current_actual_upb) exposure
from reporting_date_portfolio
group by stage, primary_stage_reason
order by stage, loans desc''')

,stage,primary_stage_reason,loans,exposure
0,1,No SICR trigger,241,"36,407,043.1800"
1,2,Relative lifetime-PD deterioration,1851,"202,281,242.8600"
2,2,Absolute lifetime-PD deterioration,1103,"148,972,769.0400"
3,2,Prior default history,219,"36,837,850.2400"
4,2,30 DPD backstop,38,"5,545,623.3000"
5,2,Modification qualitative indicator,1,"156,563.0200"
6,3,Current default/credit-impaired,18,"2,762,680.3300"


Stage 1 uses defaults in the next 12 months. Stage 2 uses remaining lifetime. Stage 3 uses discounted recovery cash shortfall.